# Chapter 7 · Measures of Association and Impact in Health: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, a register of 2,400 elective-surgery patients behind the draft of an abstract, through the research fellow's questions; only the sections with a calculation appear (7.2 and 7.4 to 7.14). **Part B** gives the written solution of every chapter exercise (1 to 16; statements are in the book) and, for the exercises whose solution needs a calculation (3, 7, 12, and 14), its code and result. The code is Python 3 with pandas. Run the cells in order.

## Part A. The case

**Data.** `data/surgery_cohort.csv`: 2,400 rows, one per patient, and 11 columns: `patient_id`, `age_years`, `sex`, `diabetes`, `smoker`, `surgery_class` (clean or contaminated), `bundle` (preoperative decolonization bundle), `ssi_30d` (surgical-site infection within 30 days), `complication_30d`, `followup_days`, and `readmitted_90d`. Each cell carries the number of the book section it belongs to.

### 7.2 The two-by-two table

*What fraction of the patients with and without diabetes had an infection?*

In [1]:
# 1. Read the surgery register: one row per patient
import pandas as pd

df = pd.read_csv("data/surgery_cohort.csv")

# 2. The two-by-two table of diabetes (rows) and infection within 30 days (columns), "yes" first
table = pd.crosstab(df["diabetes"], df["ssi_30d"])
table = table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
print(table)

# 3. The risk of infection in each row: the infections divided by the patients of the row
p1 = table.loc["yes", "yes"] / table.loc["yes"].sum()
p0 = table.loc["no", "yes"] / table.loc["no"].sum()
print("Risk with diabetes:", round(p1, 4), "; without diabetes:", round(p0, 4))

ssi_30d   yes    no
diabetes           
yes        81   519
no        111  1689
Risk with diabetes: 0.135 ; without diabetes: 0.0617


### 7.4 Risk difference

*By how much does diabetes raise the risk of infection, in absolute terms?*

In [2]:
# 1. The risk difference as a proportion, in percentage points, and per 1,000 patients
rd = p1 - p0
print("Risk difference:", round(rd, 4), "=", round(100 * rd, 2), "percentage points =", round(1000 * rd, 1), "per 1,000 patients")

Risk difference: 0.0733 = 7.33 percentage points = 73.3 per 1,000 patients


### 7.5 Risk ratio

*How many times higher is the risk, and what does the bundle do to it?*

In [3]:
# 1. The risk ratio of diabetes and the relative increase in %
rr = p1 / p0
print("Risk ratio of diabetes:", round(rr, 3), "; relative increase:", round(100 * (rr - 1), 1), "%")

# 2. The decolonization bundle: the four cells (infected and not infected, with and without the bundle)
bundle_table = pd.crosstab(df["bundle"], df["ssi_30d"])
bundle_table = bundle_table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
a = int(bundle_table.at["yes", "yes"])
b = int(bundle_table.at["yes", "no"])
c = int(bundle_table.at["no", "yes"])
d = int(bundle_table.at["no", "no"])
print("Cells a, b, c, d:", a, b, c, d)

# 3. The risk ratio of the bundle and the relative reduction in %
bundle_p1 = a / (a + b)
bundle_p0 = c / (c + d)
bundle_rr = bundle_p1 / bundle_p0
print("Risk ratio of the bundle:", round(bundle_rr, 3), "; relative reduction:", round(100 * (1 - bundle_rr), 1), "%")

Risk ratio of diabetes: 2.189 ; relative increase: 118.9 %
Cells a, b, c, d: 67 1147 125 1061
Risk ratio of the bundle: 0.524 ; relative reduction: 47.6 %


### 7.6 Prevalence ratio

*In a cross-sectional study, how do the prevalences compare?*

In [4]:
# 1. Prevalence in each shift of a cross-sectional snapshot, and the prevalence ratio
night_prevalence = 80 / 400
day_prevalence = 40 / 400
print("Prevalence ratio, night against day:", night_prevalence / day_prevalence)

Prevalence ratio, night against day: 2.0


### 7.7 Odds ratio

*How do the odds compare, and when do they differ from the risks?*

In [5]:
# 1. Risk ratio and odds ratio of diabetes for two outcomes: infection and any complication within 30 days
for outcome in ["ssi_30d", "complication_30d"]:
    outcome_table = pd.crosstab(df["diabetes"], df[outcome])
    outcome_table = outcome_table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    a = int(outcome_table.at["yes", "yes"])
    b = int(outcome_table.at["yes", "no"])
    c = int(outcome_table.at["no", "yes"])
    d = int(outcome_table.at["no", "no"])
    exposed_risk = a / (a + b)
    comparison_risk = c / (c + d)
    risk_ratio = exposed_risk / comparison_risk
    odds_ratio = (a * d) / (b * c)
    print(outcome + ": risk ratio", round(risk_ratio, 3), "; odds ratio", round(odds_ratio, 3))

ssi_30d: risk ratio 2.189 ; odds ratio 2.375
complication_30d: risk ratio 1.847 ; odds ratio 2.882


### 7.8 Rate ratio

*How do the readmission rates compare over the follow-up time?*

In [6]:
# 1. First readmissions and follow-up days in each group, and the rate per 1,000 patient-days
rates = {}
for group in ["yes", "no"]:
    part = df.loc[df["diabetes"] == group]
    events = int(part["readmitted_90d"].eq("yes").sum())
    days = int(part["followup_days"].sum())
    rates[group] = events / days
    print("Diabetes", group + ":", events, "readmissions in", days, "patient-days =", round(1000 * rates[group], 2), "per 1,000 patient-days")

# 2. The rate ratio
print("Rate ratio:", round(rates["yes"] / rates["no"], 3))

Diabetes yes: 153 readmissions in 40615 patient-days = 3.77 per 1,000 patient-days
Diabetes no: 253 readmissions in 136144 patient-days = 1.86 per 1,000 patient-days
Rate ratio: 2.027


### 7.9 Number needed to treat and number needed to harm

*How many patients must be treated to prevent one infection?*

In [7]:
# 1. The bundle: the absolute risk reduction and the number needed to treat (the reciprocal, rounded up)
import math

bundle_arr = bundle_p0 - bundle_p1
print("Absolute risk reduction:", round(bundle_arr, 4), "; number needed to treat:", math.ceil(1 / bundle_arr))

# 2. A lower-risk scenario: baseline risk of 2%, with the risk ratio of the bundle
scenario_arr = 0.02 * (1 - bundle_rr)
print("Number needed to treat at a baseline risk of 2%:", math.ceil(1 / scenario_arr))

# 3. The harm example: 20 against 10 events per 1,000, and the number needed to harm
harm_increase = 20 / 1000 - 10 / 1000
print("Number needed to harm:", math.ceil(1 / harm_increase))

Absolute risk reduction: 0.0502 ; number needed to treat: 20
Number needed to treat at a baseline risk of 2%: 105
Number needed to harm: 100


### 7.10 Attributable fraction among the exposed

*What fraction of the infections among the exposed is attributable to diabetes?*

In [8]:
# 1. Infections expected among the 600 patients with diabetes at the risk of those without, and the excess
expected_exposed = 600 * p0
excess_exposed = 81 - expected_exposed
print("Expected infections:", round(expected_exposed), "; excess infections:", round(excess_exposed))

# 2. The attributable fraction among the exposed
af_exposed = (p1 - p0) / p1
print("Attributable fraction among the exposed:", round(100 * af_exposed, 1), "%")

Expected infections: 37 ; excess infections: 44
Attributable fraction among the exposed: 54.3 %


### 7.11 Population attributable fraction

*What fraction of all the infections is attributable to diabetes?*

In [9]:
# 1. The population attributable fraction by the direct comparison and by the formula with the exposure prevalence
population_risk = 192 / 2400
exposure_prevalence = 600 / 2400
paf_direct = (population_risk - p0) / population_risk
paf_formula = exposure_prevalence * (rr - 1) / (1 + exposure_prevalence * (rr - 1))
print("Population attributable fraction, direct:", round(100 * paf_direct, 1), "%; by the formula:", round(100 * paf_formula, 1), "%")

Population attributable fraction, direct: 22.9 %; by the formula: 22.9 %


### 7.12 Stratification and confounding

*How does the association of smoking and infection behave within each surgery class?*

In [10]:
# 1. Smoking and infection within each surgery class: the four cells and the risk ratio of the class
for surgery_class, part in df.groupby("surgery_class", sort=True):
    class_table = pd.crosstab(part["smoker"], part["ssi_30d"])
    class_table = class_table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    a = int(class_table.at["yes", "yes"])
    b = int(class_table.at["yes", "no"])
    c = int(class_table.at["no", "yes"])
    d = int(class_table.at["no", "no"])
    stratum_rr = (a / (a + b)) / (c / (c + d))
    print("Class", surgery_class + ": cells", a, b, c, d, "; risk ratio", round(stratum_rr, 2))

Class clean: cells 32 570 10 458 ; risk ratio 2.49
Class contaminated: cells 29 128 121 1052 ; risk ratio 1.79


### 7.13 Simpson's paradox

*Can a crude comparison reverse when the case mix is equalized?*

In [11]:
# 1. Two hospitals: crude mortality and mortality with a common case mix (half severe, half mild)
for name, severe_n, severe_deaths, mild_n, mild_deaths in [
    ("A", 200, 40, 800, 16),
    ("B", 800, 120, 200, 2),
]:
    crude = (severe_deaths + mild_deaths) / (severe_n + mild_n)
    common_mix = 0.5 * severe_deaths / severe_n + 0.5 * mild_deaths / mild_n
    print("Hospital", name + ": crude mortality", round(100 * crude, 1), "%; with a common case mix", round(100 * common_mix, 1), "%")

Hospital A: crude mortality 5.6 %; with a common case mix 11.0 %
Hospital B: crude mortality 12.2 %; with a common case mix 8.0 %


### 7.14 What a case-control study can estimate

*What can a case-control study estimate?*

In [12]:
# 1. All the cases of infection and four controls per case drawn at random from the patients without infection (seed 7)
cases = df.loc[df["ssi_30d"] == "yes"]
noncases = df.loc[df["ssi_30d"] == "no"]
controls = noncases.sample(n=4 * len(cases), random_state=7)
case_control = pd.concat([cases, controls], ignore_index=True)

# 2. The table of diabetes and case status in the case-control sample
cc_table = pd.crosstab(case_control["diabetes"], case_control["ssi_30d"])
cc_table = cc_table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
a = int(cc_table.at["yes", "yes"])
b = int(cc_table.at["yes", "no"])
c = int(cc_table.at["no", "yes"])
d = int(cc_table.at["no", "no"])
print("Cells a, b, c, d:", a, b, c, d)

# 3. The odds ratio, and the row fractions (which are not population risks)
print("Odds ratio:", round(a * d / (b * c), 3))
print("Row fractions with diabetes and without:", round(a / (a + b), 3), round(c / (c + d), 3))

Cells a, b, c, d: 81 179 111 589
Odds ratio: 2.401
Row fractions with diabetes and without: 0.312 0.159


## Part B. Exercise solutions

### 7.17 Exercise 1. The dessert table

With exposure in the rows and illness in the first column,
$a=44$, $b=116$, $c=14$, $d=66$. The row totals are 160 and 80.
The attack risks are $44/160=27.5\%$ and $14/80=17.5\%$.
By contrast, $44/58=75.9\%$ is the fraction of ill guests who ate the
dessert. Its denominator contains cases, not everyone who ate the dessert.

### 7.17 Exercise 2. Reading a confusion matrix

|  | Condition present | Condition absent | Total |
|---|---|---|---|
| Test positive | 24 (TP) | 17 (FP) | 41 |
| Test negative | 6 (FN) | 153 (TN) | 159 |
| Total | 30 | 170 | 200 |


There are 17 false positives and 6 false negatives. Among the 30 patients
with the condition, $24/30=80\%$ test positive. Among the 41 patients with a
positive test, $24/41=58.5\%$ have the condition. The first starts with the
reference classification and the second with the test result.

### 7.17 Exercise 3. Sex and infection

The four cells, with men first, are 91, 1,118, 101, and 1,090.
Men have $91/1{,}209=7.53\%$ infection risk and women
$101/1{,}191=8.48\%$. The difference, men minus women, is
$-0.953$ percentage points: the observed risk is slightly lower in men.
The following code reproduces the risks and difference.
The printed negative difference describes this comparison without establishing
that sex itself accounts for it.

In [13]:
# 1. Infection risk by sex, in %: men and women
sex_table = pd.crosstab(df["sex"], df["ssi_30d"])
sex_table = sex_table.reindex(index=["M", "F"], columns=["yes", "no"])
sex_risks = sex_table["yes"] / sex_table.sum(axis=1)
print("Risk in men and in women (%):", (100 * sex_risks).round(2).tolist())

# 2. The risk difference between men and women, in percentage points
print("Risk difference, men minus women:", round(100 * (sex_risks["M"] - sex_risks["F"]), 3), "percentage points")

Risk in men and in women (%): [7.53, 8.48]
Risk difference, men minus women: -0.953 percentage points


### 7.17 Exercise 4. Risk ratios

The trial risks are $100/2{,}000=0.05$ and $150/2{,}000=0.075$:
$\mathrm{RD}=-0.025$, $\mathrm{RR}=2/3\approx0.667$, and
$\mathrm{RRR}=1/3\approx33.3\%$ over two years.
For the dessert, $\mathrm{RR}=0.275/0.175\approx1.57$, exceeding the
seafood comparison's 1.33. The dessert also has the larger absolute difference,
10 versus approximately 7 percentage points. Guests may have eaten both dishes;
joint exposure, timing, symptoms, and relevant laboratory findings are needed
to investigate causation. The larger ratio alone does not identify the culprit.

### 7.17 Exercise 5. Prevalence

The prevalences are $90/300=30\%$ and $45/300=15\%$, giving PR $=2$ and a
15 percentage-point difference. A matching sentence is: “At assessment, back
pain was present in 30% of ward nurses and 15% of outpatient nurses, a
prevalence ratio of 2.” The survey did not ascertain new-onset back pain over
follow-up.

### 7.17 Exercise 6. Risk and odds

For risks 0.40 and 0.20, RR $=2$ and
OR $=(0.40/0.60)/(0.20/0.80)=2.667$.
The factor is $0.80/0.60=1.333$, so OR exceeds RR by 33.3%.
For risks 0.04 and 0.02, RR $=2$ and
OR $=(0.04/0.96)/(0.02/0.98)=2.0417$.
The factor is $0.98/0.96=1.02083$, a 2.08% relative excess. The latter
approximation is closer; whether that discrepancy is acceptable depends on
the intended use.

### 7.17 Exercise 7. Rates

The dressing rates are $30/1{,}500\times1{,}000=20$ and
$20/2{,}500\times1{,}000=8$ infections per 1,000 catheter-days.
The rate ratio is $20/8=2.5$ and the difference is 12 infections per
1,000 catheter-days. For the bundle comparison, the next block totals
readmissions and time at risk in each group.
The observed rate ratio is 0.85. Neither set of crude rates establishes an
intervention effect: eligibility, prognosis, selection, and observation
processes must support a causal comparison.

In [14]:
# 1. First readmissions and follow-up days with and without the bundle
bundle_rates = {}
for group in ["yes", "no"]:
    part = df.loc[df["bundle"] == group]
    events = int(part["readmitted_90d"].eq("yes").sum())
    days = int(part["followup_days"].sum())
    bundle_rates[group] = events / days
    print("Bundle", group + ":", events, "readmissions in", days, "patient-days")

# 2. The rate ratio of the bundle
print("Rate ratio of the bundle:", round(bundle_rates["yes"] / bundle_rates["no"], 3))

Bundle yes: 192 readmissions in 90760 patient-days
Bundle no: 214 readmissions in 85999 patient-days
Rate ratio of the bundle: 0.85


### 7.17 Exercise 8. Baseline risk and NNT

The trial ARR is $0.075-0.05=0.025$, giving NNT $=40$ over two years.
Holding RR at $2/3$ when the comparator risk is 0.02 gives treatment risk
$0.01\overline{3}$, ARR $0.00\overline{6}$, and NNT $=150$.
The second calculation assumes the same causal relative effect transports
to the new population. Lower baseline risk gives less absolute benefit under
that assumption.

### 7.17 Exercise 9. Harm

The risks are $18/900=0.02$ and $6/900=0.00\overline{6}$.
The increase is $12/900=0.01\overline{3}$, or 1.33 percentage points.
NNH $=900/12=75$ over one year for major bleeding. Comparing this number with
an NNT for stroke over two years also requires comparing the periods, the
clinical consequences of the two outcomes, and their uncertainty.

### 7.17 Exercise 10. Exposed attributable fraction

The unexposed risk is $10/200=0.05$. If it applies to the 200 exposed
patients under removal of exposure, the expected count is 10 instead of 30,
a difference of 20. The exposed fraction is $20/30=66.7\%$.
Calling these infections preventable asserts a causal intervention comparison:
the specified removal would give those patients the comparison risk. The
observed table alone supplies the association, not that assertion.

### 7.17 Exercise 11. Population fraction

The cohort RR is $(81/600)/(111/1{,}800)=2.\overline{189}$.
At 10% exposure prevalence,

$$ \mathrm{PAF}=\frac{0.10(2.\overline{189}-1)}{1+0.10(2.\overline{189}-1)}
=\frac{22}{207}\approx10.6\%. $$

The population fraction is lower than 22.9% because fewer people carry the
exposure under the fixed-effect scenario. This calculation retains the
unexposed-risk substitution and transport assumptions.

### 7.17 Exercise 12. Stratifying diabetes

The next block computes class-specific risks and their ratios, then reports
the share of clean operations by diabetes status.
The clean-operation counts are 259 of 600 patients with diabetes and 811 of
1,800 without it. Their proportions, 43.2% and 45.1%, are much closer
than the smoking proportions of 79.3% and 28.5%. The stratum RRs are also
close to the crude RR of 2.19. Agreement does not prove the absence of
confounding by other variables, or establish a causal interpretation.

In [15]:
# 1. The risk ratio of diabetes within each surgery class
for surgery_class, part in df.groupby("surgery_class", sort=True):
    tab = pd.crosstab(part["diabetes"], part["ssi_30d"])
    risks = tab["yes"] / tab.sum(axis=1)
    print("Class", surgery_class + ": risk ratio of diabetes", round(risks["yes"] / risks["no"], 3))

# 2. The percentage of clean operations among patients with and without diabetes
for group in ["yes", "no"]:
    part = df.loc[df["diabetes"] == group]
    print("Diabetes", group + ": clean operations", round(100 * part["surgery_class"].eq("clean").mean(), 1), "%")

Class clean: risk ratio of diabetes 2.129
Class contaminated: risk ratio of diabetes 2.158
Diabetes yes: clean operations 43.2 %
Diabetes no: clean operations 45.1 %


### 7.17 Exercise 13. A common hospital case mix

Using 25% severe and 75% mild patients, A has standardized risk
$0.25(0.20)+0.75(0.02)=6.5\%$ and B has
$0.25(0.15)+0.75(0.01)=4.5\%$.
B remains lower at this common case mix, whereas its observed crude risk
was higher, 12.2% against 5.6%. Actual and common-weight comparisons answer
different descriptive questions. Broad severity categories may still hide
prognostic differences, and selection, ascertainment, and uncertainty matter
before inferring a causal difference in care quality.

### 7.17 Exercise 14. Two controls per case

The next block repeats the draw with a different control-to-case ratio.
The OR is 2.130 rather than 2.401 because the smaller random control sample
has a different exposure composition. The sampled row fractions rise to
45.3% and 28.0% because fewer noncases accompany the same cases.
The population target of the OR remains the same under the sampling
assumptions, but a finite estimate is not algebraically fixed at that target.

In [16]:
# 1. All the cases and two controls per case, drawn at random from the patients without infection (seed 7)
cases = df.loc[df["ssi_30d"] == "yes"]
noncases = df.loc[df["ssi_30d"] == "no"]
controls = noncases.sample(n=2 * len(cases), random_state=7)
selected = pd.concat([cases, controls], ignore_index=True)

# 2. The table of diabetes and case status
tab = pd.crosstab(selected["diabetes"], selected["ssi_30d"])
a = int(tab.at["yes", "yes"])
b = int(tab.at["yes", "no"])
c = int(tab.at["no", "yes"])
d = int(tab.at["no", "no"])
print("Cells a, b, c, d:", a, b, c, d)

# 3. The odds ratio and the row fractions
print("Odds ratio:", round(a * d / (b * c), 3))
print("Row fractions with diabetes and without:", round(a / (a + b), 3), round(c / (c + d), 3))

Cells a, b, c, d: 81 98 111 286
Odds ratio: 2.13
Row fractions with diabetes and without: 0.453 0.28


### 7.17 Exercise 15. Rewriting a relative benefit

RR $=1-0.40=0.60$. The other risk is $0.005\times0.60=0.003$,
so ARR $=0.002$ and the causally interpreted NNT is 500 over two years.
A report can say: “Two-year stroke risk was 0.3% on the new treatment
and 0.5% on the comparator, corresponding to 3 versus 5 strokes per
1,000 patients, an absolute reduction of 0.2 percentage points and a
relative reduction of 40%.” Group counts, the study design, and an
appropriate uncertainty interval are needed to assess the estimate's precision
and interpretation.

### 7.17 Exercise 16. Choosing a measure

(a) A causally supported absolute risk reduction times the number of eligible
operations estimates the expected change in infection count; check that the
population, protocol, and horizon match, and include implementation costs.
(b) A rate ratio compares events per time at risk; inspect censoring and whether
observation differs with prognosis. If the decision needs fixed-time risk,
use a suitable survival estimate instead.
(c) A prevalence ratio and prevalence difference compare existing hypertension;
check that assessment and population eligibility are comparable.
(d) A population attributable fraction describes the proportional reduction
under the specified intervention; assess confounding, the intervention's
feasibility, and uncertainty rather than equating association with prevention.

---
*Synthetic data (`generate_data.py`, seed 7). Teaching values only; nothing here is an estimate for a real hospital.*